In [7]:
import os
import re
import pandas as pd
from functools import reduce
import json

In [8]:
# Get working directory
working_dir = os.getcwd()
print(f"Working directory: {working_dir}")

Working directory: /home/samuele/Documents/uni_projects/hateradar/Annotators_workload/PILOT


- - -
### Comments disagreement

In [9]:

input_dir = "VideosComments/youtube/annotated_comments"

for newspaper in ["corriere_della_sera", "repubblica"]:

    newspaper_dir = os.path.join(input_dir, newspaper)

    # Group files by video_id
    video_groups = {}

    for file_name in os.listdir(newspaper_dir):

        if file_name.endswith(".csv"):

            # Example: abc123_ann1.csv -> video_id = abc123
            match = re.match(r"(.+)_ann\d+\.csv", file_name)

            if match:
                video_id = match.group(1)

                if video_id not in video_groups:
                    video_groups[video_id] = []

                video_groups[video_id].append(file_name)

    # Process each video
    for video_id, files in video_groups.items():

        if len(files) < 2:
            continue

        annotator_dfs = []

        for idx, file_name in enumerate(sorted(files)):

            file_path = os.path.join(newspaper_dir, file_name)

            df = pd.read_csv(file_path)

            # Keep needed columns
            df = df[["comment_id", "text", "label"]].copy()

            # Rename label column per annotator
            df = df.rename(columns={"label": f"label_ann{idx+1}"})

            # Keep text only once
            if idx > 0:
                df = df.drop(columns=["text"])

            annotator_dfs.append(df)

        # Merge all annotator files on comment_id
        merged_df = reduce(
            lambda left, right: pd.merge(
                left,
                right,
                on="comment_id",
                how="inner"
            ),
            annotator_dfs
        )

        # Find disagreement rows
        label_cols = [col for col in merged_df.columns if col.startswith("label_")]

        disagreement_mask = (
            merged_df[label_cols]
            .nunique(axis=1) > 1
        )

        disagreements = merged_df[disagreement_mask]

        if not disagreements.empty:

            print(f"\n{'='*80}")
            print(f"Disagreements found for video: {video_id}")
            print(f"{'='*80}")

            for _, row in disagreements.iterrows():

                print(f"\nComment ID: {row['comment_id']}")
                print(f"Text: {row['text']}")

                for col in label_cols:
                    print(f"{col}: {row[col]}")

                print("-" * 80)


Disagreements found for video: CAVns3bw4tc

Comment ID: Ugy9uks04AyWeHq503Z4AaABAg
Text: Cercate di finirla un euro di basilico fate pena
label_ann1: 1
label_ann2: 0
label_ann3: 0
label_ann4: 1
--------------------------------------------------------------------------------

Comment ID: UgwPEZnkIg4-JFPCQKl4AaABAg.AWrg3rBhoprAWrvovumvPE
Text: Se non puoi permettertelo non vai, e tranquillo, non chiude perché c'è chi ci va
label_ann1: 0
label_ann2: 0
label_ann3: 1
label_ann4: 0
--------------------------------------------------------------------------------

Comment ID: UgzGWnzObwWjHYs1X-l4AaABAg.AWrvXDqSzP5AWxNRXWLQIZ
Text: Esatto e chiudessero tutte le pizzerie al piatto che mettono 1 euro per una foglia di basilico
label_ann1: 0
label_ann2: 0
label_ann3: 0
label_ann4: 1
--------------------------------------------------------------------------------

Comment ID: UgzGWnzObwWjHYs1X-l4AaABAg.AWrvXDqSzP5AWxYWB3egUG
Text: Ti do una notizia il basilico sulla Margherita è uno dei pochi ingr

### Label disagreement

In [6]:
import os
import re
import json
import pandas as pd
from collections import defaultdict

input_dir = "VideosComments/youtube/annotated_metadata"

for newspaper in ["corriere_della_sera", "repubblica"]:

    newspaper_dir = os.path.join(input_dir, newspaper)

    # Group files by video_id
    video_groups = defaultdict(list)

    for file_name in os.listdir(newspaper_dir):

        if file_name.endswith(".json"):

            # Example: abc123_ann1.json -> video_id = abc123
            match = re.match(r"(.+)_ann\d+\.json", file_name)

            if match:
                video_id = match.group(1)
                video_groups[video_id].append(file_name)

    # Process each video
    for video_id, files in video_groups.items():

        if len(files) < 2:
            continue

        annotations = []

        for file_name in sorted(files):

            file_path = os.path.join(newspaper_dir, file_name)

            with open(file_path, "r", encoding="utf-8") as f:
                data = json.load(f)

            annotator_number = data.get("annotator_number", "unknown")
            topic = data.get("topic", "MISSING")

            annotations.append({
                "annotator": annotator_number,
                "topic": topic,
                "title": data.get("title", "")
            })

        # Check disagreement
        unique_topics = set(a["topic"] for a in annotations)

        if len(unique_topics) > 1:

            print(f"\n{'='*80}")
            print(f"Topic disagreement found for video: {video_id}")
            print(f"{'='*80}")

            print(f"Title: {annotations[0]['title']}")

            for ann in annotations:
                print(
                    f"Annotator {ann['annotator']}: "
                    f"{ann['topic']}"
                )

            print("-" * 80)


Topic disagreement found for video: CAVns3bw4tc
Title: Un euro in più per il basilico sulla pizza Margherita: polemica a Bari
Annotator 1: lifestyle and leisure
Annotator 2: economy, business and finance
Annotator 3: lifestyle and leisure
--------------------------------------------------------------------------------
